In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q01/rewritten/o4_mini_high/checkpoints/post_cell_0_try_0.pickle

trying: ['load_lineitem']
me:  1
trying: ['DATA_ROOT']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['tpch_parent']
me:  0
trying: ['pd']
me:  0
trying: ['lineitem']


me:  1


Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable pd
Declaring variable load_lineitem
Declaring variable lineitem


In [4]:
%%RecordEvent
%%time
### cell 1 ###

# 1) Filter by ship date
date = pd.Timestamp("1998-09-02")
cols = [
    "L_QUANTITY",
    "L_EXTENDEDPRICE",
    "L_DISCOUNT",
    "L_TAX",
    "L_RETURNFLAG",
    "L_LINESTATUS",
    "L_SHIPDATE",
    "L_ORDERKEY",
]
lineitem_filtered = lineitem.loc[:, cols]
lineitem_filtered = lineitem_filtered[lineitem_filtered.L_SHIPDATE <= date]

# 2) Compute line‐level measures
lineitem_filtered["AVG_QTY"] = lineitem_filtered.L_QUANTITY
lineitem_filtered["AVG_PRICE"] = lineitem_filtered.L_EXTENDEDPRICE
lineitem_filtered["DISC_PRICE"] = (
    lineitem_filtered.L_EXTENDEDPRICE * (1 - lineitem_filtered.L_DISCOUNT)
)
lineitem_filtered["CHARGE"] = (
    lineitem_filtered.L_EXTENDEDPRICE
    * (1 - lineitem_filtered.L_DISCOUNT)
    * (1 + lineitem_filtered.L_TAX)
)

# 3) Group and aggregate
gb = lineitem_filtered.groupby(
    ["L_RETURNFLAG", "L_LINESTATUS"], as_index=False
)[
    [
        "L_QUANTITY",
        "L_EXTENDEDPRICE",
        "DISC_PRICE",
        "CHARGE",
        "AVG_QTY",
        "AVG_PRICE",
        "L_DISCOUNT",
        "L_ORDERKEY",
    ]
]
total = gb.agg(
    {
        "L_QUANTITY": "sum",
        "L_EXTENDEDPRICE": "sum",
        "DISC_PRICE": "sum",
        "CHARGE": "sum",
        "AVG_QTY": "mean",
        "AVG_PRICE": "mean",
        "L_DISCOUNT": "mean",
        "L_ORDERKEY": "count",
    }
)

# 4) Match pandas ordering and dtypes
total = total.sort_values(["L_RETURNFLAG", "L_LINESTATUS"])\
             .reset_index(drop=True)
total = total.astype({
    "L_QUANTITY": "float64",
    "L_EXTENDEDPRICE": "float64",
    "DISC_PRICE": "float64",
    "CHARGE": "float64",
    "AVG_QTY": "float64",
    "AVG_PRICE": "float64",
    "L_DISCOUNT": "float64",
    "L_ORDERKEY": "int64",
})

CPU times: user 91.3 ms, sys: 66.3 ms, total: 158 ms
Wall time: 165 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q01/rewritten/o4_mini_high/checkpoints/post_cell_1_try_3.pickle

migration speed (bps): 826116472.4505768
---------------------------
variables to migrate:
gb 48
DATA_ROOT 80
date 48
lineitem 48
total 48
STORAGE_OPTS 64
tpch_parent 54
load_lineitem 144
lineitem_filtered 48
cols 120
pd 72
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q01/rewritten/o4_mini_high/checkpoints/post_cell_1_try_3.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['date', 'lineitem_filtered', 'total', 'cols', 'gb']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q01/opt_cell_exec_info_1_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[1], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q01/annotated/checkpoints/post_cell_1.pickle
assert compare_df(total_opt, total)

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
